# Week 13: Standing on Giants — Running Pretrained Vision Models from the Hugging Face Hub

**This is the final lab.** All semester you built vision tools from the ground up — filters, features, even a Transformer by hand. This week we flip it around: instead of building, we **download models other people already trained on web-scale data** and just run them. That is how real computer-vision work actually happens.

One library does all of it — **Hugging Face `transformers`**, backed by the **Hugging Face Hub** (hundreds of thousands of pretrained models). For most tasks a single helper, `pipeline`, runs any of them in about three lines. Watch for the pattern that repeats in every section:

> **load a model -> call it on an image -> read the output.**

`pipeline` is just the one-line version of that pattern, and the only thing that changes from task to task is the *shape of the output*. Once you can picture that, you can pick up almost any vision model on the Hub and run it yourself — which is exactly the goal this whole course has been building toward.

Today's tour: **classification** (and swapping models), **zero-shot** classification (CLIP), **object detection** (YOLO), **segment-everything** (SAM), and **image captioning** (BLIP).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch

%matplotlib inline

### Environment setup

Run this once. On Colab it installs Hugging Face `transformers` and downloads the images we use. PyTorch is already available on Colab.

The first time you load each model it **downloads the weights** and caches them — later calls are instant. The models range from small (YOLOS ~ 130 MB) to large (CLIP ~ 600 MB, BLIP ~ 990 MB). Everything runs on **CPU**; no GPU needed. One slow spot: **SAM's segment-everything cell (Section 4) takes about a minute on CPU** — that is expected.

In [ ]:
import os, sys, subprocess, urllib.request

IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, "__IPYTHON__") else False

if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers"])
    REPO_URL = "https://raw.githubusercontent.com/HyeongminLEE/image-processing-tutorial/main"
    os.makedirs("images", exist_ok=True)
    for fname in ["parrots_square.jpg", "mot_color70.jpg", "mot_color83.jpg", "einstein_monroe.jpg"]:
        if not os.path.exists(f"images/{fname}"):
            urllib.request.urlretrieve(f"{REPO_URL}/images/{fname}", f"images/{fname}")
            print(f"Downloaded {fname}")
    IMG_DIR = "images/"
else:
    IMG_DIR = "../images/"

print(f"Running on: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Image directory: {IMG_DIR}")
print(f"PyTorch version: {torch.__version__}")

### Display helpers

`show_image` draws one image; `show_scores` draws one horizontal bar chart of label -> score (reused for classification top-k and CLIP probabilities); `show_detections` draws an object detector's boxes; `show_masks` overlays segmentation masks. As always, each helper shows exactly one figure — call it again for another.

In [ ]:
# Each helper shows exactly one figure.
from matplotlib.patches import Rectangle


def show_image(img, title=None, scale=4):
    fig, ax = plt.subplots(figsize=(scale, scale))
    if img.ndim == 2:
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    else:
        ax.imshow(img)
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_scores(labels, scores, title=None, scale=(6, 3)):
    # labels: list of strings; scores: matching list of floats in [0, 1]
    fig, ax = plt.subplots(figsize=scale)
    positions = range(len(labels))
    ax.barh(positions, scores, color="#3b82f6")
    ax.set_yticks(positions)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()              # first entry on top
    ax.set_xlim(0, 1)
    ax.set_xlabel("score")
    for i in range(len(scores)):
        ax.text(scores[i], i, f" {scores[i]:.2f}", va="center")
    if title:
        ax.set_title(title)
    plt.tight_layout()
    plt.show()


def show_detections(img, detections, title=None, scale=6):
    # detections: list of {'label', 'score', 'box': {xmin, ymin, xmax, ymax}}
    img = np.asarray(img)
    fig, ax = plt.subplots(figsize=(scale, scale))
    ax.imshow(img)
    for d in detections:
        box = d["box"]
        x, y = box["xmin"], box["ymin"]
        w = box["xmax"] - box["xmin"]
        h = box["ymax"] - box["ymin"]
        ax.add_patch(Rectangle((x, y), w, h, fill=False, edgecolor="#ef4444", linewidth=2))
        ax.text(x, y - 4, f"{d['label']} {d['score']:.2f}",
                color="white", fontsize=9,
                bbox=dict(facecolor="#ef4444", edgecolor="none", pad=1))
    if title:
        ax.set_title(title)
    ax.axis("off")
    plt.tight_layout()
    plt.show()


def show_masks(img, masks, title=None, scale=6):
    # masks: list of 2-D boolean arrays (same H x W as img), one per segment
    img = np.asarray(img)
    fig, ax = plt.subplots(figsize=(scale, scale))
    ax.imshow(img)
    overlay = np.zeros((img.shape[0], img.shape[1], 4))
    cmap = plt.cm.nipy_spectral
    for i in range(len(masks)):
        color = cmap((i + 1) / (len(masks) + 1))
        overlay[masks[i]] = (color[0], color[1], color[2], 0.5)
    ax.imshow(overlay)
    if title:
        ax.set_title(f"{title} ({len(masks)} masks)")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

---

## 0. The Hugging Face Hub & `pipeline` — 3 Lines to a Prediction

The **Hugging Face Hub** hosts hundreds of thousands of pretrained models, and `pipeline` is the highest-level way to run one: pick a **task** and a **model**, and it handles all the pre- and post-processing for you. Here we ask for image classification and point it at **ViT** (`google/vit-base-patch16-224`), a Vision Transformer trained on ImageNet.

🤗 Model card: [google/vit-base-patch16-224](https://huggingface.co/google/vit-base-patch16-224)

In [ ]:
img = Image.open(IMG_DIR + "parrots_square.jpg").convert("RGB")
print("image size (W, H):", img.size)
show_image(np.array(img), title="input image")

In [ ]:
from transformers import pipeline

# Downloads the ViT weights the first time this runs.
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

In [ ]:
results = classifier(img)

# The output is a list of dicts, each {'label': str, 'score': float}, best first.
print("type:", type(results))
print("number of results:", len(results))
print("results[0]:", results[0])

In [ ]:
top_labels = []
top_scores = []
for r in results:
    top_labels.append(r["label"])
    top_scores.append(r["score"])

show_scores(top_labels, top_scores, title="ViT top-5")

Three lines did it: load the image, build the `pipeline`, call it. The result is just a list of `{'label', 'score'}` dicts — that list is the whole interface. And as the next section shows, the *same three lines* drive any other classifier on the Hub.

---

## 1. Same 3 Lines, a Different Model

The Hub has **thousands** of image classifiers. Swapping the checkpoint string is all it takes to run a completely different architecture — the calling code never changes. Let's send the same parrots image through a **ResNet** (a classic CNN) and a **ConvNeXt** (a modern CNN) and compare their top guess with ViT's.

🤗 Model cards: [microsoft/resnet-50](https://huggingface.co/microsoft/resnet-50) · [facebook/convnext-tiny-224](https://huggingface.co/facebook/convnext-tiny-224)

In [ ]:
resnet = pipeline("image-classification", model="microsoft/resnet-50")
resnet_results = resnet(img)
print("ResNet-50 top-1:", resnet_results[0])

In [ ]:
convnext = pipeline("image-classification", model="facebook/convnext-tiny-224")
convnext_results = convnext(img)
print("ConvNeXt-tiny top-1:", convnext_results[0])

In [ ]:
print("ViT       :", results[0]["label"], f"({results[0]['score']:.2f})")
print("ResNet-50 :", resnet_results[0]["label"], f"({resnet_results[0]['score']:.2f})")
print("ConvNeXt  :", convnext_results[0]["label"], f"({convnext_results[0]['score']:.2f})")

Three different model families, the **same output structure**, the same three lines — only the checkpoint string changed. Each model's **model card** on huggingface.co lists its size, training data, and accuracy, so you can trade off speed against quality. Browse the Hub, copy a model id, run it.

### Try together — run another model from the Hub

It's the last lab, so there are no graded exercises — we run everything together. Let's pick another image classifier from **huggingface.co/models** (filter by the **Image Classification** task) and compare its **top-1** with ViT's. Here we use a **Swin Transformer** (`microsoft/swin-tiny-patch4-window7-224`, a different architecture); try swapping in another id such as `google/vit-large-patch16-224` and rerun.

🤗 Model cards: [microsoft/swin-tiny-patch4-window7-224](https://huggingface.co/microsoft/swin-tiny-patch4-window7-224) · [google/vit-large-patch16-224](https://huggingface.co/google/vit-large-patch16-224)

In [ ]:
# Input image
show_image(np.array(img), title="img")

# Same one line as Section 0 — only the model id changed.
my_clf = pipeline("image-classification", model="microsoft/swin-tiny-patch4-window7-224")
my_results = my_clf(img)

my_labels = []
my_scores = []
for r in my_results:
    my_labels.append(r["label"])
    my_scores.append(r["score"])

show_scores(my_labels, my_scores, title="Swin-tiny top-5")

print("Swin-tiny top-1:", my_results[0]["label"])
print("ViT-base  top-1:", results[0]["label"])

---

## 2. Zero-Shot Classification with CLIP

ViT and ResNet are locked to the **1000 fixed ImageNet classes** they were trained on. Change the label set and you must retrain. **CLIP** removes that ceiling: it learned to place images and text in **one shared space**, so it can score an image against *any* labels you write — with **no training**. The Hub exposes this as the `zero-shot-image-classification` task; we use **CLIP** (`openai/clip-vit-base-patch32`).

🤗 Model card: [openai/clip-vit-base-patch32](https://huggingface.co/openai/clip-vit-base-patch32)

In [ ]:
zero_shot = pipeline("zero-shot-image-classification", model="openai/clip-vit-base-patch32")

In [ ]:
candidate_labels = ["a parrot", "a dog", "a cat", "an airplane"]
results_zs = zero_shot(img, candidate_labels=candidate_labels)

# Again a list of {'label', 'score'} dicts, best first — scores are probabilities over your labels.
print("type:", type(results_zs))
print("results_zs[0]:", results_zs[0])

In [ ]:
labels_zs = []
scores_zs = []
for r in results_zs:
    labels_zs.append(r["label"])
    scores_zs.append(r["score"])

show_scores(labels_zs, scores_zs, title="CLIP zero-shot")

CLIP scored the image against each phrase and turned the similarities into probabilities — the closest meaning wins. Nothing was trained; swap `candidate_labels` for anything you like and rerun. (Under the hood `pipeline` wraps each label in a prompt like `"This is a photo of a parrot."` — a small trick that, the slides noted, measurably boosts accuracy.)

### Try together — zero-shot with our own labels

Let's run zero-shot on the street image (`img_street`) with labels we choose — some that *are* in the scene (a bus, a car, a building) and some that are *not* (a beach, a forest) — to see the contrast. Then a quick **prompt-engineering** check: CLIP lets us pass `hypothesis_template` (the sentence each label is dropped into, default `"This is a photo of {}."`); we swap the template and watch whether the scores move.

In [ ]:
# Input image
img_street = Image.open(IMG_DIR + "mot_color70.jpg").convert("RGB")
show_image(np.array(img_street), title="img_street")

my_street_labels = ["a bus", "a car", "a building", "a beach", "a forest"]
out = zero_shot(img_street, candidate_labels=my_street_labels)

disp_labels = []
disp_scores = []
for r in out:
    disp_labels.append(r["label"])
    disp_scores.append(r["score"])

show_scores(disp_labels, disp_scores, title="CLIP zero-shot (street scene)")

In [ ]:
# Same labels, a different prompt template -> do the scores move?
out2 = zero_shot(img_street, candidate_labels=my_street_labels,
                 hypothesis_template="a blurry photo of a {}.")
for r in out2:
    print(f"{r['score']:.3f}  {r['label']}")

---

## 3. Object Detection — YOLO

Classification gives one label for the whole image. **Detection** finds *where* each object is and draws a box around it. We use **YOLOS** (`hustvl/yolos-tiny`) — a ViT adapted for detection, so it ties straight back to last week. Same `pipeline` call, new task: `object-detection`.

🤗 Model card: [hustvl/yolos-tiny](https://huggingface.co/hustvl/yolos-tiny)

In [ ]:
img_street = Image.open(IMG_DIR + "mot_color70.jpg").convert("RGB")

detector = pipeline("object-detection", model="hustvl/yolos-tiny")
detections = detector(img_street, threshold=0.8)

# A list of dicts; each has a label, a confidence score, and a pixel box.
print("number of detections:", len(detections))
print("detections[0]:", detections[0])

In [ ]:
show_detections(img_street, detections, title="YOLOS object detection")

Each detection is `{'label', 'score', 'box': {xmin, ymin, xmax, ymax}}` — the box is in pixel coordinates, ready to draw. Raise or lower `threshold` to keep fewer or more boxes. (YOLOS is the Hub-native YOLO; the production-standard **Ultralytics YOLO** lives in its own library and follows the same idea — load weights, call on an image, read boxes.)

### Try together — detect and count on a new image

Let's run the detector on a second street image (`img_street2`), draw the boxes, and **count** how many of one class (here `"car"`) it found. Change the class or the `threshold` and rerun to watch the count move.

In [ ]:
# Input image
img_street2 = Image.open(IMG_DIR + "mot_color83.jpg").convert("RGB")
show_image(np.array(img_street2), title="img_street2")

my_detections = detector(img_street2, threshold=0.8)
show_detections(img_street2, my_detections, title="detections on img_street2")

# Count how many cars were found.
car_count = 0
for d in my_detections:
    if d["label"] == "car":
        car_count += 1
print("cars found:", car_count)
print("total detections:", len(my_detections))

---

## 4. Segment Everything — SAM

Detection draws boxes; **segmentation** labels every pixel. **SAM** (Segment Anything Model, `facebook/sam-vit-base`) is *class-agnostic* and *promptable*: in "segment everything" mode it finds and outlines **every** distinct region on its own — no labels, no training. The Hub task is `mask-generation`.

🤗 Model card: [facebook/sam-vit-base](https://huggingface.co/facebook/sam-vit-base)

> **Heads up:** this is the slow cell — about a minute on CPU. Run it once and wait.

In [ ]:
segmenter = pipeline("mask-generation", model="facebook/sam-vit-base")

# points_per_side controls how densely SAM probes the image (lower = faster, fewer masks).
sam_output = segmenter(img, points_per_side=16)

# A dict: 'masks' is a list of boolean H x W arrays (one per segment); 'scores' rates each.
print("keys:", list(sam_output.keys()))
print("number of masks:", len(sam_output["masks"]))
print("one mask shape:", sam_output["masks"][0].shape, "dtype:", sam_output["masks"][0].dtype)

In [ ]:
show_masks(img, sam_output["masks"], title="SAM segment-everything")

SAM carved the image into regions without being told what any of them are — that is the "anything" in Segment Anything. It does not *name* the segments (no "parrot" label); pair it with a classifier or a text prompt when you need names. SAM can also take a **point or box prompt** ("segment *this*"), which is how it is most often used in practice.

---

## 5. Bonus: Image Captioning — BLIP

Every task so far returned labels or boxes. **Captioning** returns a **sentence**: a model that *describes* the image in natural language. **BLIP** (`Salesforce/blip-image-captioning-base`) does exactly this.

🤗 Model card: [Salesforce/blip-image-captioning-base](https://huggingface.co/Salesforce/blip-image-captioning-base)

There is no one-line `pipeline` for captioning, so we drop one level down to the model itself — still high-level: a **processor** prepares the image, the model **generates** token ids, and the processor **decodes** them back into text. (That processor -> model -> decode shape is exactly what `pipeline` was doing for us all along.)

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

In [ ]:
inputs = blip_processor(images=img, return_tensors="pt")
generated_ids = blip_model.generate(**inputs, max_new_tokens=30)

# generate() returns token ids; the processor decodes them back into a string.
print("generated_ids shape:", generated_ids.shape)
caption = blip_processor.decode(generated_ids[0], skip_special_tokens=True)
print("caption:", caption)

In [ ]:
img_face = Image.open(IMG_DIR + "einstein_monroe.jpg").convert("RGB")
show_image(np.array(img_face), title="input")

inputs = blip_processor(images=img_face, return_tensors="pt")
generated_ids = blip_model.generate(**inputs, max_new_tokens=30)
print("caption:", blip_processor.decode(generated_ids[0], skip_special_tokens=True))

Same idea, a brand-new kind of output — free-form text. From a single label to a box to a full mask to a sentence, the only thing that changed across five tasks was the *shape of what came back*.

---

### Wrap-up

- One library — **Hugging Face `transformers`** — drove **five** different models: ViT / ResNet / ConvNeXt (classification), CLIP (zero-shot), YOLOS (detection), SAM (segmentation), and BLIP (captioning).
- The pattern barely changed: **load a model -> call it -> read the output.** Four tasks needed only a one-line `pipeline`; captioning peeked one level under the hood (**processor -> model -> decode**) — which is exactly what `pipeline` wraps. Only the **shape of the output** (a label list, a box list, a mask list, a sentence) really differed.
- That is the whole course's destination: take an open-source vision model off the Hub, understand its inputs and outputs, run inference, and adapt it to your own problem — from **building** these tools by hand to **standing on** the giants who pretrained them.